# 🎤 YingMusic-SVC — Google Colab

เปลี่ยนเสียงร้องในเพลงให้เป็นเสียงของคนอื่น (ใช้แค่ตัวอย่างเสียง 10–25 วินาที)

1. `Runtime → Change runtime type → T4 GPU`
2. กด ▶ ที่ Cell ด้านล่าง
3. รอติดตั้ง (ครั้งแรก ~5–8 นาที) → จะมีปุ่ม **Choose Files** ให้อัปโหลดเพลง (และเสียงเป้าหมายถ้าเลือก `upload`)
4. รอแปลงเสร็จ → ฟังผลได้ใน notebook และไฟล์จะดาวน์โหลดให้อัตโนมัติ

จะแปลงเพลงถัดไป กด ▶ ซ้ำได้เลย (ไม่ต้องติดตั้งใหม่)

> ทำงานใน notebook ทั้งหมด ไม่เปิดหน้าเว็บ — ใช้กับ Colab ฟรีได้โดยไม่ผิดนโยบาย

In [ ]:
#@title ▶ แปลงเสียงเพลง
#@markdown **เสียงเป้าหมาย** — `upload` = อัปโหลดเสียงร้อง/พูดที่สะอาด ไม่มีดนตรี ยาว 10–25 วินาที
TIMBRE = "upload" #@param ["upload", "prompts/tp.wav", "prompts/syz.wav"]
#@markdown ติ๊กถ้าเพลงมีดนตรีประกอบ (จะแยกเสียงร้องออกก่อน) — เอาติ๊กออกถ้าเป็นไฟล์เสียงร้องล้วน
HAS_ACCOMPANIMENT = True #@param {type:"boolean"}
#@markdown ติ๊กถ้าอยากเก็บเสียงคอรัสของต้นฉบับไว้ในดนตรีตอนมิกซ์กลับ
KEEP_BACKING_VOCAL = False #@param {type:"boolean"}
#@markdown คุณภาพ vs ความเร็ว: 100 = ค่าแนะนำ, 30–50 = เร็วขึ้น
DIFFUSION_STEPS = 100 #@param {type:"slider", min:10, max:200, step:10}
#@markdown ติ๊กถ้าต้องการใช้ไฟล์เพลงและเสียงร้องที่อัปโหลด/แยกไว้แล้วจากรอบล่าสุด (ไม่ต้องอัปโหลดใหม่)
REUSE_LAST_RUN = False #@param {type:"boolean"}
#@markdown ---
#@markdown ขั้นสูง (ปกติไม่ต้องแก้)
REPO_URL = "https://github.com/GiantAILab/YingMusic-SVC" #@param {type:"string"}
BRANCH = "" #@param {type:"string"}
HF_TOKEN = "" #@param {type:"string"}

import os, sys, shlex, shutil, subprocess, time, glob
from google.colab import files
from IPython.display import Audio, display

def sh(cmd, cwd=None):
    """รันคำสั่ง, แสดง output สด, หยุดถ้าล้มเหลว"""
    p = subprocess.Popen(
        cmd,
        shell=True,
        cwd=cwd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True
    )
    for line in p.stdout:
        print(line, end="")
    if p.wait() != 0:
        raise SystemExit(f"❌ ล้มเหลว (exit {p.returncode}): {cmd}")

def ensure_compat_patches():
    """ฝัง Patch แก้ปัญหา torchaudio.sox_effects, torch.load และ scipy บน Python 3.13 / Colab"""
    import torchaudio
    ta_dir = os.path.dirname(torchaudio.__file__)
    sox_file = os.path.join(ta_dir, "sox_effects.py")
    init_file = os.path.join(ta_dir, "__init__.py")

    sox_code = """import os, tempfile, subprocess
import numpy as np
import soundfile as sf
import torch

def init_sox_effects():
    pass

def shutdown_sox_effects():
    pass

def effect_names():
    return ["echo", "reverb", "gain", "norm", "highpass", "lowpass", "equalizer", "rate", "channels"]

def apply_effects_tensor(tensor, sample_rate, effects, channels_first=True):
    if not effects:
        return tensor, sample_rate
    device, dtype = tensor.device, tensor.dtype
    wav = tensor.detach().cpu().float()
    if wav.ndim == 1:
        wav = wav.unsqueeze(0)
    wav_np = wav.transpose(0, 1).numpy() if channels_first else wav.numpy()

    with tempfile.TemporaryDirectory() as tmpdir:
        in_wav = os.path.join(tmpdir, "in.wav")
        out_wav = os.path.join(tmpdir, "out.wav")
        sf.write(in_wav, wav_np, int(sample_rate), subtype="FLOAT")
        flat_effects = [str(x) for eff in effects for x in eff]
        cmd = ["sox", "-q", in_wav, out_wav] + flat_effects
        res = subprocess.run(cmd, capture_output=True, text=True)
        if res.returncode != 0 or not os.path.exists(out_wav):
            cur_in = in_wav
            for idx, eff in enumerate(effects):
                step_out = os.path.join(tmpdir, f"step_{idx}.wav")
                r = subprocess.run(["sox", "-q", cur_in, step_out] + [str(x) for x in eff], capture_output=True, text=True)
                if r.returncode == 0 and os.path.exists(step_out):
                    cur_in = step_out
            out_wav = cur_in
        out_np, out_sr = sf.read(out_wav, dtype="float32", always_2d=True)

    out_tensor = torch.from_numpy(out_np)
    if channels_first:
        out_tensor = out_tensor.transpose(0, 1).contiguous()
    return out_tensor.to(device=device, dtype=dtype), int(out_sr)

def apply_effects_file(path, effects, normalize=True, channels_first=True, format=None):
    data, sr = sf.read(str(path), dtype="float32", always_2d=True)
    tensor = torch.from_numpy(data)
    if channels_first:
        tensor = tensor.transpose(0, 1).contiguous()
    return apply_effects_tensor(tensor, sr, effects, channels_first=channels_first)
"""
    with open(sox_file, "w") as f:
        f.write(sox_code)

    patch_marker = "# --- YINGMUSIC_COLAB_PATCH ---"
    with open(init_file, "r") as f:
        init_content = f.read()

    if patch_marker not in init_content:
        init_append = f"""
{patch_marker}
from . import sox_effects
import torch as _torch
import soundfile as _sf
import numpy as _np

if not hasattr(_np, "int"): _np.int = int
if not hasattr(_np, "float"): _np.float = float
if not hasattr(_np, "complex"): _np.complex = complex

try:
    import scipy.signal as _sig
    import scipy.signal.windows as _win
    if not hasattr(_sig, "hann") and hasattr(_win, "hann"):
        _sig.hann = _win.hann
except Exception:
    pass

_orig_torch_load = _torch.load
def _compat_torch_load(*args, **kwargs):
    if "weights_only" not in kwargs or kwargs["weights_only"] is None:
        kwargs["weights_only"] = False
    return _orig_torch_load(*args, **kwargs)
_torch.load = _compat_torch_load

_orig_ta_load = load
def _compat_ta_load(filepath, *args, **kwargs):
    try:
        return _orig_ta_load(filepath, *args, **kwargs)
    except Exception:
        data, sr = _sf.read(str(filepath), dtype="float32", always_2d=True)
        channels_first = kwargs.get("channels_first", True)
        t = _torch.from_numpy(data)
        if channels_first:
            t = t.transpose(0, 1).contiguous()
        return t, int(sr)
load = _compat_ta_load

_orig_ta_save = save
def _compat_ta_save(filepath, src, sample_rate, *args, **kwargs):
    try:
        return _orig_ta_save(filepath, src, sample_rate, *args, **kwargs)
    except Exception:
        channels_first = kwargs.get("channels_first", True)
        wav = src.detach().cpu().float()
        if wav.ndim == 1:
            wav = wav.unsqueeze(0)
        wav_np = wav.transpose(0, 1).numpy() if channels_first else wav.numpy()
        _sf.write(str(filepath), wav_np, int(sample_rate))
save = _compat_ta_save
"""
        with open(init_file, "a") as f:
            f.write(init_append)

q = shlex.quote
REPO_DIR = "/content/YingMusic-SVC"
READY = "/content/.ymsvc_ready_v3"

if subprocess.run(["nvidia-smi", "-L"], capture_output=True).returncode != 0:
    raise SystemExit("❌ ไม่พบ GPU — Runtime → Change runtime type → เลือก T4 GPU แล้วรันใหม่")

# ================= 1. ติดตั้ง (ครั้งแรกครั้งเดียว) =================
if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    sh(f"git clone -q --depth 1 {'-b ' + q(BRANCH) if BRANCH else ''} {q(REPO_URL)} {REPO_DIR}")
else:
    sh(f"git -C {REPO_DIR} pull -q --ff-only")
os.chdir(REPO_DIR)
sh('git log -1 --format="code version: %h %s"')

if not os.path.exists(READY):
    print("📦 กำลังติดตั้ง dependencies ... (ข้อความเตือนของ pip ข้ามได้)")
    sh("apt-get -qq update > /dev/null && apt-get -qq install -y sox libsox-fmt-all ffmpeg > /dev/null")
    sh(
        'pip install -q "librosa>=0.10.2,<1" "transformers>=4.46.3,<5" "huggingface-hub>=0.28.1,<1.0" '
        'munch einops "descript-audio-codec==1.0.0" pydub soundfile pyyaml '
        'omegaconf ml_collections loralib wandb beartype "rotary_embedding_torch==0.3.5" '
        'matplotlib scipy tqdm audiomentations pedalboard'
    )

    print("⬇️ กำลังโหลดโมเดล ...")
    from huggingface_hub import hf_hub_download, snapshot_download
    token = HF_TOKEN.strip() or None

    os.makedirs("checkpoints", exist_ok=True)
    os.makedirs("accom_separation/ckpt/bs_roformer", exist_ok=True)

    hf_hub_download("GiantAILab/YingMusic-SVC", "YingMusic-SVC-full.pt", local_dir="checkpoints", token=token)
    hf_hub_download("GiantAILab/YingMusic-SVC", "bs_roformer.ckpt", local_dir="accom_separation/ckpt/bs_roformer", token=token)

    # โมเดลเสริม: รองรับทั้ง HF cache, โฟลเดอร์ checkpoints และ path/to
    rmvpe_cached = hf_hub_download("lj1995/VoiceConversionWebUI", "rmvpe.pt", cache_dir="./checkpoints", token=token)
    camp_cached = hf_hub_download("funasr/campplus", "campplus_cn_common.bin", cache_dir="./checkpoints", token=token)
    if not os.path.exists("checkpoints/rmvpe.pt"):
        shutil.copy(rmvpe_cached, "checkpoints/rmvpe.pt")
    if not os.path.exists("checkpoints/campplus_cn_common.bin"):
        shutil.copy(camp_cached, "checkpoints/campplus_cn_common.bin")

    snapshot_download("nvidia/bigvgan_v2_44khz_128band_512x", cache_dir="./checkpoints/hf_cache",
                      allow_patterns=["config.json", "bigvgan_generator.pt"], token=token)
    snapshot_download("nvidia/bigvgan_v2_44khz_128band_512x", local_dir="./nvidia/bigvgan_v2_44khz_128band_512x",
                      allow_patterns=["config.json", "bigvgan_generator.pt"], token=token)
    snapshot_download("openai/whisper-small", cache_dir="./checkpoints/hf_cache",
                      allow_patterns=["*.json", "*.safetensors", "*.bin", "*.txt"], token=token)
    snapshot_download("openai/whisper-small", local_dir="./openai/whisper-small",
                      allow_patterns=["*.json", "*.safetensors", "*.bin", "*.txt"], token=token)

    open(READY, "w").close()
    print("✅ ติดตั้งเสร็จสมบูรณ์")

# บังคับตรวจสอบและฝัง Patch ทุกครั้งเผื่อรันต่อจาก Session เดิม
ensure_compat_patches()
os.makedirs("path", exist_ok=True)
if not os.path.exists("path/to"):
    os.symlink(os.path.abspath("checkpoints"), "path/to")

sh("python -c \"import torch, torchaudio, librosa, transformers, dac; from torchaudio import sox_effects; "
   "print('torch', torch.__version__, '| CUDA', torch.cuda.is_available(), '| sox_effects OK')\"")

SVC_CKPT = os.path.abspath("checkpoints/YingMusic-SVC-full.pt")
SEP_CKPT = os.path.abspath("accom_separation/ckpt/bs_roformer/bs_roformer.ckpt")
SEP_CFG = os.path.abspath("accom_separation/ckpt/bs_roformer/config_bd_roformer.yaml")

# ================= 2. อัปโหลดไฟล์ & 3. แยกเสียงร้อง/ดนตรี =================
prev_runs = sorted(glob.glob(os.path.abspath("colab_runs/*")))
if REUSE_LAST_RUN and prev_runs:
    WORK = prev_runs[-1]
    RUN_ID = os.path.basename(WORK)
    print(f"\n♻️ ใช้ไฟล์จากรอบล่าสุด: {WORK}")
    uploaded_files = glob.glob(os.path.join(WORK, "upload_*"))
    song_title = os.path.splitext(os.path.basename(uploaded_files[0]))[0].replace("upload_", "") if uploaded_files else "song"
    SONG_WAV = os.path.join(WORK, "song.wav")
    TIMBRE_WAV = os.path.join(WORK, "timbre.wav") if TIMBRE == "upload" else os.path.abspath(TIMBRE)
    stems = os.path.join(WORK, "sep_out", "song")
    if HAS_ACCOMPANIMENT and os.path.exists(os.path.join(stems, "vocals.wav")):
        SOURCE_VOCAL = os.path.join(stems, "vocals.wav")
        ACCOMPANY = os.path.join(stems, "instrumental.wav")
        if KEEP_BACKING_VOCAL and os.path.exists(os.path.join(stems, "backing_vocal.wav")):
            mixed_acc = os.path.join(stems, "instrumental_with_backing.wav")
            if not os.path.exists(mixed_acc):
                sh(f"ffmpeg -hide_banner -loglevel error -y -i {q(ACCOMPANY)} -i {q(os.path.join(stems, 'backing_vocal.wav'))} "
                   f"-filter_complex amix=inputs=2:normalize=0 {q(mixed_acc)}")
            ACCOMPANY = mixed_acc
    else:
        SOURCE_VOCAL, ACCOMPANY = SONG_WAV, ""
else:
    RUN_ID = time.strftime("%Y%m%d_%H%M%S")
    WORK = os.path.abspath(f"colab_runs/{RUN_ID}")
    os.makedirs(WORK, exist_ok=True)

    def upload_one(prompt):
        print(f"\n{prompt}")
        up = files.upload()
        if not up:
            raise SystemExit("❌ ไม่ได้อัปโหลดไฟล์")
        name = next(iter(up))
        safe_name = os.path.basename(name).replace(" ", "_")
        dst = os.path.join(WORK, "upload_" + safe_name)
        shutil.move(name, dst)
        return dst, os.path.splitext(safe_name)[0]

    song_raw, song_title = upload_one("🎵 อัปโหลดเพลงต้นฉบับ (mp3/wav/flac/m4a):")
    SONG_WAV = os.path.join(WORK, "song.wav")
    sh(f"ffmpeg -hide_banner -loglevel error -y -i {q(song_raw)} -ar 44100 {q(SONG_WAV)}")

    if TIMBRE == "upload":
        timbre_raw, _ = upload_one("🗣️ อัปโหลดเสียงเป้าหมาย (เสียงร้อง/พูดสะอาด 10–25 วินาที):")
        TIMBRE_WAV = os.path.join(WORK, "timbre.wav")
        sh(f"ffmpeg -hide_banner -loglevel error -y -i {q(timbre_raw)} -ar 44100 -ac 1 {q(TIMBRE_WAV)}")
    else:
        TIMBRE_WAV = os.path.abspath(TIMBRE)

    if HAS_ACCOMPANIMENT:
        print("\n🎚️ กำลังแยกเสียงร้องออกจากดนตรี ...")
        sep_in, sep_out = os.path.join(WORK, "sep_in"), os.path.join(WORK, "sep_out")
        os.makedirs(sep_in, exist_ok=True)
        shutil.copy(SONG_WAV, sep_in)
        sh(f"PYTHONWARNINGS=ignore python inference.py --model_type bs_roformer --config_path {q(SEP_CFG)} "
           f"--start_check_point {q(SEP_CKPT)} --input_folder {q(sep_in)} --store_dir {q(sep_out)} --disable_detailed_pbar",
           cwd="accom_separation")
        stems = os.path.join(sep_out, "song")
        SOURCE_VOCAL = os.path.join(stems, "vocals.wav")
        ACCOMPANY = os.path.join(stems, "instrumental.wav")
        if KEEP_BACKING_VOCAL:
            backing_wav = os.path.join(stems, "backing_vocal.wav")
            if os.path.exists(backing_wav):
                mixed_acc = os.path.join(stems, "instrumental_with_backing.wav")
                sh(f"ffmpeg -hide_banner -loglevel error -y -i {q(ACCOMPANY)} -i {q(backing_wav)} "
                   f"-filter_complex amix=inputs=2:normalize=0 {q(mixed_acc)}")
                ACCOMPANY = mixed_acc
    else:
        SOURCE_VOCAL, ACCOMPANY = SONG_WAV, ""

# ================= 4. แปลงเสียง =================
print("\n🎤 กำลังแปลงเสียง ... (เพลงยาว 3–4 นาทีบน T4 ใช้เวลาหลายนาที)")
EXP = f"colab_{RUN_ID}"
accompany_arg = f"--accompany {q(ACCOMPANY)}" if ACCOMPANY else ""
sh(f"python -u my_inference.py --source {q(SOURCE_VOCAL)} --target {q(TIMBRE_WAV)} "
   f"--diffusion-steps {int(DIFFUSION_STEPS)} --checkpoint {q(SVC_CKPT)} --expname {q(EXP)} --cuda 0 --fp16 True "
   f"{accompany_arg} --config configs/YingMusic-SVC.yml")

vc_files = sorted(glob.glob(f"outputs/{EXP}/*.wav"))
mix_files = sorted(glob.glob(f"outputs/{EXP}/accompany/*.wav"))
if not vc_files:
    raise SystemExit("❌ ไม่พบไฟล์ผลลัพธ์ — ดู error ด้านบน")

# ================= 5. ฟัง & ดาวน์โหลด =================
results = [("เสียงร้องที่แปลงแล้ว", vc_files[-1], f"{song_title}_vocal.wav")]
if mix_files:
    results.insert(0, ("เพลงที่มิกซ์กับดนตรีแล้ว", mix_files[-1], f"{song_title}_cover.wav"))
for label, wav, nice_name in results:
    final = os.path.join(WORK, nice_name.replace(" ", "_"))
    shutil.copy(wav, final)
    preview = os.path.splitext(final)[0] + ".mp3"
    sh(f"ffmpeg -hide_banner -loglevel error -y -i {q(final)} -b:a 192k {q(preview)}")
    print(f"\n✅ {label}: {final}")
    display(Audio(filename=preview))
    files.download(final)